# Notebook 1 — Three structural limitations of OLS on time series

This notebook demonstrates the three limitations of ordinary least squares on time series
identified in *Symbolic Pure Time* (Vázquez Broquá, 2026) and shows how SPTLS removes each
one with the same closed-form solve on the kinematic jet.

**Limitations addressed:**
1. Pre-testing requirement: OLS needs a Dickey–Fuller step to select the specification
2. Residuals as nuisance: the distributional character of innovations is not read from the fit
3. Two procedures: predictable component and innovations recovered separately

All results are fully reproducible. Requires: `numpy`, `scipy`, `matplotlib`, and the `gsd`
library in `lib/` (pure NumPy).


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), '..', '..', '..', 'lib'))
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import gsd

rng = np.random.default_rng(42)
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})


## Limitation 1 — Pre-testing requirement

Standard OLS needs to know the integration order *before* the fit. We demonstrate with an
I(1) AR(2) process: the naive AR(1) misses the unit root; detecting it requires a
Dickey–Fuller test, which then determines the specification.

**SPTLS fix:** integration order is encoded in the eigenvalue spectrum of M̂ and read
algebraically after the fit — no pre-test required.


In [ ]:
# DGP: I(1) AR(2)  y_t = 1.6 y_{t-1} - 0.6 y_{t-2} + eps_t
# characteristic roots: {1.0, 0.6}
T = 600
phi1, phi2 = 1.6, -0.6
y = np.zeros(T)
for t in range(2, T):
    y[t] = phi1*y[t-1] + phi2*y[t-2] + rng.standard_normal()

# 1a. Naive OLS AR(1)
Y, X_ar1 = y[1:], np.column_stack([np.ones(T-1), y[:-1]])
b_ar1 = np.linalg.lstsq(X_ar1, Y, rcond=None)[0]
resid_ar1 = Y - X_ar1 @ b_ar1
dw_ar1 = np.sum(np.diff(resid_ar1)**2) / np.sum(resid_ar1**2)

# 1b. Textbook AR(2) (after Dickey-Fuller directs us here)
Y2, X_ar2 = y[2:], np.column_stack([np.ones(T-2), y[1:-1], y[:-2]])
b_ar2 = np.linalg.lstsq(X_ar2, Y2, rcond=None)[0]
resid_ar2 = Y2 - X_ar2 @ b_ar2
dw_ar2 = np.sum(np.diff(resid_ar2)**2) / np.sum(resid_ar2**2)
r2_ar2 = 1 - np.var(resid_ar2) / np.var(Y2)

# 1c. SPTLS — no pre-test
spt = gsd.SPTLS()
res = spt.fit(y)
eigvals = np.linalg.eigvals(res.M[:3, :3])
mods = np.sort(np.abs(eigvals))[::-1]

print(f"AR(1) naive:   DW = {dw_ar1:.3f}  (target ≈ 2.0; far below → autocorrelated residuals)")
print(f"AR(2) correct: DW = {dw_ar2:.3f}  R² = {r2_ar2:.4f}")
print(f"SPTLS M̂ eigenvalue moduli: {mods[0]:.4f}, {mods[1]:.4f}, {mods[2]:.4f}")
print(f"  → largest ≈ 1.0 = unit root;  second ≈ 0.6 = stationary root")
print(f"  Integration order read algebraically. No Dickey–Fuller step used.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].plot(y, color='#4f46e5', lw=0.8, alpha=0.9)
axes[0].set_title('Series: I(1) AR(2)', fontweight='bold')
axes[0].set_xlabel('t'); axes[0].set_ylabel('y_t')

# unit circle with eigenvalues
theta = np.linspace(0, 2*np.pi, 300)
axes[1].plot(np.cos(theta), np.sin(theta), 'k-', lw=0.5, alpha=0.3)
axes[1].axhline(0, color='#ccc', lw=0.5); axes[1].axvline(0, color='#ccc', lw=0.5)
for ev in eigvals:
    axes[1].scatter(ev.real, ev.imag, s=90, zorder=5,
                    color='#4f46e5', edgecolors='white', linewidths=1.5)
axes[1].set_xlim(-1.3, 1.3); axes[1].set_ylim(-1.3, 1.3)
axes[1].set_aspect('equal')
axes[1].set_title('Eigenvalues of M̂  (unit circle = integration boundary)', fontweight='bold')
axes[1].set_xlabel('Re'); axes[1].set_ylabel('Im')

plt.tight_layout()
plt.savefig('fig_lim1_integration_order.png', dpi=130, bbox_inches='tight')
plt.show()
print("Largest eigenvalue modulus:", mods[0].round(4), " — on the unit circle → I(1)")


## Limitation 2 — Residuals as nuisance

In standard OLS the residuals are inspected after the fit to validate assumptions (Ljung–Box,
Jarque–Bera, etc.). Their distributional character is not a primary output of the estimation.

**SPTLS fix:** the one-step residuals of SPTLS are the Wold innovation sequence of the
kinematic embedding, orthogonal to M̂ by the OLS projection theorem. Their distribution is
read as a first-class dynamic quantity — specifically via the excess kurtosis κ of the
velocity-row residuals — rather than validated as a by-product.

We demonstrate with two series: Gaussian AR(1) and a heavy-tail AR(1). Standard OLS
identifies both as "AR(1) with autocorrelated errors" on the residual level row; the
innovation channel κ distinguishes them.


In [ ]:
T = 500

# Series A: Gaussian AR(1)  ρ = 0.7
y_gauss = np.zeros(T)
for t in range(1, T):
    y_gauss[t] = 0.7*y_gauss[t-1] + rng.standard_normal()

# Series B: AR(1) ρ = 0.7, but t(3) innovations (heavy tail)
nu = 3.0
def sample_t(n, nu, rng):
    z = rng.standard_normal(n)
    chi = rng.chisquare(nu, n)
    return z / np.sqrt(chi / nu)

y_heavy = np.zeros(T)
innov_heavy = sample_t(T, nu, rng)
for t in range(1, T):
    y_heavy[t] = 0.7*y_heavy[t-1] + innov_heavy[t]

# Fit SPTLS on both
res_g = gsd.SPTLS().fit(y_gauss)
res_h = gsd.SPTLS().fit(y_heavy)

# velocity-row residuals (row index 1 in the 3-dim jet)
vel_resid_g = res_g.resid_jet[:, 1]
vel_resid_h = res_h.resid_jet[:, 1]

kappa_g = stats.kurtosis(vel_resid_g, fisher=True)   # excess kurtosis
kappa_h = stats.kurtosis(vel_resid_h, fisher=True)

print(f"Gaussian AR(1) — κ (excess kurtosis of velocity residuals) = {kappa_g:.3f}  (≈ 0 for Gaussian)")
print(f"Heavy-tail AR(1) — κ = {kappa_h:.3f}  (>> 0: heavy-tail innovations)")
print()
print("Both series have similar AR(1) R² and DW on the level row.")
print("The innovation channel κ distinguishes them in a single read.")

r2_g = list(res_g.rsquared.values())[0]
r2_h = list(res_h.rsquared.values())[0]
print(f"  Gaussian  R² = {r2_g:.4f}")
print(f"  Heavy-tail R² = {r2_h:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

# residual distributions
axes[0].hist(vel_resid_g, bins=40, color='#4f46e5', alpha=0.7, density=True)
xg = np.linspace(-4, 4, 200)
axes[0].plot(xg, stats.norm.pdf(xg), 'k--', lw=1.5, label='N(0,1)')
axes[0].set_title(f'Gaussian AR(1)\nκ = {kappa_g:.2f}', fontweight='bold')
axes[0].set_xlabel('velocity-row residual')
axes[0].legend(fontsize=9)

axes[1].hist(vel_resid_h, bins=40, color='#e0552b', alpha=0.7, density=True)
axes[1].plot(xg, stats.norm.pdf(xg, scale=vel_resid_h.std()), 'k--', lw=1.5, label='N(0,σ)')
axes[1].set_title(f'Heavy-tail AR(1)\nκ = {kappa_h:.2f}', fontweight='bold')
axes[1].set_xlabel('velocity-row residual')
axes[1].legend(fontsize=9)

# M-I summary
axes[2].bar(['Gaussian', 'Heavy-tail'], [kappa_g, kappa_h],
            color=['#4f46e5', '#e0552b'], alpha=0.85, width=0.4)
axes[2].axhline(0, color='#ccc', lw=1)
axes[2].set_ylabel('κ  (excess kurtosis of innovations)')
axes[2].set_title('Innovation channel κ\n(same solve that delivers M̂)', fontweight='bold')

plt.tight_layout()
plt.savefig('fig_lim2_innovation_channel.png', dpi=130, bbox_inches='tight')
plt.show()


## Limitation 3 — Two procedures for two objects

Classical regression recovers the predictable component from the coefficient vector and
the innovations from the residual series — two steps, two objects.

**SPTLS fix:** the M-I decomposition delivers both simultaneously as two orthogonal channels
of the same closed-form solve.

- **Memory axis Δ** = ‖M̂ − M̂^iid‖_F, where M̂^iid is the SPTLS operator on a shuffled
  version of the series (baseline that absorbs only the mechanical autocorrelation from
  differencing). Δ measures genuine temporal dependence.
- **Innovation axis κ** = excess kurtosis of velocity-row residuals.

The two are orthogonal by the OLS projection theorem. We demonstrate on four synthetic
processes that occupy distinct regions of the M-I plane.


In [ ]:
def sptls_mi(y, n_shuffles=8, seed=0):
    """Compute M-I decomposition: (delta, kappa) for a univariate series y."""
    res = gsd.SPTLS().fit(y)
    M = res.M
    # iid baseline: average M over shuffled copies
    rng_s = np.random.default_rng(seed)
    M_iid = np.zeros_like(M)
    for _ in range(n_shuffles):
        ys = y.copy(); rng_s.shuffle(ys)
        M_iid += gsd.SPTLS().fit(ys).M / n_shuffles
    delta = np.linalg.norm(M - M_iid, 'fro')
    kappa = stats.kurtosis(res.resid_jet[:, 1], fisher=True)
    return delta, kappa

T = 500
processes = {
    'White noise':       lambda: rng.standard_normal(T),
    'AR(1) ρ=0.7':       lambda: np.array([y:=np.zeros(T)] and [y.__setitem__(t, 0.7*y[t-1]+rng.standard_normal()) for t in range(1,T)] and y)[0] if False else _ar1(T, 0.7),
    'Random walk I(1)':  lambda: np.cumsum(rng.standard_normal(T)),
    'Heavy-tail shock':  lambda: _heavy(T),
}

def _ar1(T, rho):
    y = np.zeros(T)
    for t in range(1, T): y[t] = rho*y[t-1] + rng.standard_normal()
    return y

def _heavy(T):
    y = np.zeros(T)
    innov = sample_t(T, 3.0, rng)
    for t in range(1, T): y[t] = 0.3*y[t-1] + 0.4*innov[t]
    return y

results = {}
for name, gen in [('White noise', lambda: rng.standard_normal(T)),
                  ('AR(1) ρ=0.7', lambda: _ar1(T, 0.7)),
                  ('Random walk I(1)', lambda: np.cumsum(rng.standard_normal(T))),
                  ('Heavy-tail shock', lambda: _heavy(T))]:
    deltas, kappas = [], []
    for rep in range(8):
        y_rep = gen()
        d, k = sptls_mi(y_rep, seed=rep)
        deltas.append(d); kappas.append(k)
    results[name] = (np.array(deltas), np.array(kappas))
    print(f"{name:25s}  Δ = {np.mean(deltas):.3f} ± {np.std(deltas):.3f}   "
          f"κ = {np.mean(kappas):.2f} ± {np.std(kappas):.2f}")


In [ ]:
COLORS = {'White noise':'#9aa0ab','AR(1) ρ=0.7':'#4f46e5',
           'Random walk I(1)':'#1f9d6b','Heavy-tail shock':'#e0552b'}

fig, ax = plt.subplots(figsize=(7, 5.5))
for name, (deltas, kappas) in results.items():
    ax.scatter(kappas, deltas, s=70, color=COLORS[name], alpha=0.85,
               label=name, zorder=5, edgecolors='white', linewidths=0.8)
ax.axhline(0, color='#ddd', lw=1); ax.axvline(0, color='#ddd', lw=1)
ax.set_xlabel('κ  (innovation axis — excess kurtosis of velocity residuals)')
ax.set_ylabel('Δ  (memory axis — Frobenius from i.i.d. baseline)')
ax.set_title('M-I plane\nFour processes, one closed-form solve', fontweight='bold')
ax.legend(frameon=False, fontsize=10)
plt.tight_layout()
plt.savefig('fig_lim3_mi_plane.png', dpi=130, bbox_inches='tight')
plt.show()
print("\nM-I plane separates processes by their dynamic structure.")
print("Both channels come from the same OLS solve — not two separate procedures.")
